# 04a - SCD Type 2: Initial Backfill & Demo Extract Generation

**Project:** Retail Analytics & Product Dimension History

## ⚠️ RUN THIS MANUALLY, ONCE ONLY - NEVER SCHEDULE THIS NOTEBOOK
This notebook performs a full RESET of the product dimension (Cell 2 uses
mode("overwrite")) and generates the simulated "day 2" extract used to
demonstrate SCD Type 2. In a real production system, this notebook's Cell 2
represents a one-time historical backfill - the kind of operation a real
data engineer runs manually, exactly once, completely separate from any
recurring schedule, since running it again would destroy accumulated history.

The actual recurring, schedulable, idempotent SCD2 logic lives in
04b_retail_scd2_merge - THAT notebook is what's included in this project's
scheduled Job, not this one.

## What this notebook does
1. Loads the initial product snapshot (backdated to a 2000-01-01 sentinel
   date, so it correctly predates all real transaction history)
2. Generates a simulated "day 2" extract (price changes, category change,
   discontinued product, new products) as a real CSV file
3. Ingests that extract into its own isolated bronze table

## Tables/files created
- `main.retail_analytics.silver_dim_products_scd2` (full reset)
- `/Volumes/main/retail_analytics/raw_data/Products_day2.csv`
- `main.retail_analytics.bronze_products_day2_extract`

In [0]:
from pyspark.sql.functions import col, lit, current_date, expr

# Reads ONLY from bronze_products, which under this design will only ever
# contain the original Products.csv - the day-2 extract is never mixed in.
bronze_products = spark.table("main.retail_analytics.bronze_products")

initial_products = (
    bronze_products
    .withColumnRenamed("ProductID", "product_id")
    .withColumnRenamed("ProductName", "product_name")
    .withColumnRenamed("Category", "category")
    .withColumnRenamed("SubCategory", "subcategory")
    .withColumnRenamed("UnitPrice", "unit_price")
    .withColumnRenamed("CostPrice", "cost_price")
    .withColumn("product_sk", expr("uuid()"))
    .withColumn("effective_start_date", lit("2000-01-01").cast("date"))  # sentinel "beginning of history" date
    .withColumn("effective_end_date", lit(None).cast("date"))
    .withColumn("is_current", lit(True))
)

(
    initial_products.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("main.retail_analytics.silver_dim_products_scd2")
)

print("Row count:", spark.table("main.retail_analytics.silver_dim_products_scd2").count())

Row count: 50


## Test data generation (simulating a "day 2" extract)

Everything in this cell exists ONLY to create realistic test data. In a real
company, this cell wouldn't exist - a new file would simply arrive from the
source system. The actual SCD2 merge logic (further down) is completely
generic and contains no hardcoded product IDs.

In [0]:
from pyspark.sql.functions import when, col, lit, round as _round
from pyspark.sql import Row

current_products = spark.table("main.retail_analytics.silver_dim_products_scd2").filter(col("is_current") == True)

DISCONTINUED_PRODUCT_ID = "P015"

changed_products = (
    current_products
    .filter(col("product_id") != DISCONTINUED_PRODUCT_ID)
    .withColumn("unit_price", _round(when(col("product_id").isin("P001", "P002", "P003"), col("unit_price") * 1.10).otherwise(col("unit_price")), 2))
    .withColumn("category", when(col("product_id") == "P010", lit("Reclassified Category")).otherwise(col("category")))
    .select("product_id", "product_name", "category", "subcategory", "unit_price", "cost_price")
)

new_products = spark.createDataFrame([
    Row(product_id="P051", product_name="Wireless Earbuds Pro", category="Electronics", subcategory="Audio", unit_price=89.99, cost_price=42.00),
    Row(product_id="P052", product_name="Insulated Water Bottle", category="Home & Kitchen", subcategory="Drinkware", unit_price=24.99, cost_price=9.50),
])

simulated_new_extract_df = changed_products.unionByName(new_products)

(
    simulated_new_extract_df.coalesce(1).write
    .format("csv")
    .option("header", "true")
    .mode("overwrite")
    .save("/Volumes/main/retail_analytics/raw_data/products_day2_extract_temp")
)

files_in_temp = dbutils.fs.ls("/Volumes/main/retail_analytics/raw_data/products_day2_extract_temp")
part_file = [f.path for f in files_in_temp if f.name.startswith("part-")][0]
dbutils.fs.cp(part_file, "/Volumes/main/retail_analytics/raw_data/Products_day2.csv", recurse=False)
dbutils.fs.rm("/Volumes/main/retail_analytics/raw_data/products_day2_extract_temp", recurse=True)

print("Wrote Products_day2.csv - simulated new extract landed as a real file")

Wrote Products_day2.csv - simulated new extract landed as a real file


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

day2_schema = StructType([
    StructField("product_id", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("subcategory", StringType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("cost_price", DoubleType(), True),
])

day2_extract = (
    spark.read
    .option("header", "true")
    .schema(day2_schema)
    .csv("/Volumes/main/retail_analytics/raw_data/Products_day2.csv")
)

(
    day2_extract.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("main.retail_analytics.bronze_products_day2_extract")
)

print("Row count:", spark.table("main.retail_analytics.bronze_products_day2_extract").count())

Row count: 51


In [0]:
new_product_extract_df = (
    spark.table("main.retail_analytics.bronze_products_day2_extract")
    .dropDuplicates(["product_id"])  # defensive safety net, even though overwrite mode should prevent duplicates
)

new_product_extract_df.createOrReplaceTempView("new_product_extract")
print("Rows in new extract:", new_product_extract_df.count())

Rows in new extract: 51
